In [1]:
import numpy as np
import pandas as pd
import pyterrier as pt
import os
import ir_datasets
from urllib.parse import urlparse, parse_qs
import re


In [2]:
if not pt.started():
    pt.init()

/tmp/ipykernel_82191/3057724015.py:1: DeprecationWarning: Call to deprecated function (or staticmethod) started. (use pt.java.started() instead) -- Deprecated since version 0.11.0.
  if not pt.started():
Java started and loaded: pyterrier.java, pyterrier.terrier.java [version=5.11 (build: craig.macdonald 2025-01-13 21:29), helper_version=0.0.8]
/tmp/ipykernel_82191/3057724015.py:2: DeprecationWarning: Call to deprecated method pt.init(). Deprecated since version 0.11.0.
java is now started automatically with default settings. To force initialisation early, run:
pt.java.init() # optional, forces java initialisation
  pt.init()


In [3]:
dataset = ir_datasets.load('istella22/test')

In [4]:
INDEX_EXISTS = True
index_dir = '/media/ersel/Expansion/istella22_index3'

if INDEX_EXISTS:
    index = pt.IndexFactory.of(index_dir)
else:
    def doc_to_dict_generator(docs):
        global error_no
        for doc in docs:
            try:
                yield {"docno": doc.doc_id, "text": doc.text}
            except:
                yield {"docno": str(error_no), "text": "istella document error"}
                error_no -= 1
    
    indexer = pt.index.IterDictIndexer(index_dir)
    index = indexer.index(doc_to_dict_generator(dataset.docs))


In [5]:
bm25_retriever = pt.BatchRetrieve(index, wmodel="BM25") % 100

/tmp/ipykernel_82191/3716249422.py:1: DeprecationWarning: Call to deprecated class BatchRetrieve. (use pt.terrier.Retriever() instead) -- Deprecated since version 0.11.0.
  bm25_retriever = pt.BatchRetrieve(index, wmodel="BM25") % 100


In [6]:
search_res = bm25_retriever.search("sad life")
print(search_res)
print(len(search_res))

   qid    docid             docno  rank      score     query
0    1  2775598  1990011300499881     0  23.154382  sad life
1    1  1935575  1990010900176593     1  23.025193  sad life
2    1  3032524  1990011401260999     2  22.925853  sad life
3    1   406548  1990010102281031     3  22.835376  sad life
4    1  1021330  1990010401842904     4  22.354891  sad life
..  ..      ...               ...   ...        ...       ...
95   1   857857  1990010302557020    95  19.702936  sad life
96   1  1222075  1990010501697919    96  19.694342  sad life
97   1   716068  1990010300627743    97  19.647187  sad life
98   1  2889559  1990011302137931    98  19.636935  sad life
99   1  3323228  1990011502613031    99  19.631194  sad life

[100 rows x 6 columns]
100


In [7]:
queries = {}
relevant_set = set()
for query in dataset.queries:
    queries[query.query_id] = {"text": query.text, "docs":{}}
for qrel in dataset.qrels:
    queries[qrel.query_id]['docs'][qrel.doc_id] = qrel.relevance
    relevant_set.add(qrel.doc_id)


In [10]:
def dcg_at_k(relevances, k=None):
    if k:
        relevances = relevances[:k]
    return sum([rel / np.log2(i + 1) for i, rel in enumerate(relevances, 1)])

def ndcg_at_k(relevances, k=None):
    dcg = dcg_at_k(relevances, k)
    idcg = dcg_at_k(sorted(relevances, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0

In [9]:
queries2 = {} 
for qid, q_dict in queries.items():
    # qid, q_dict = '263', queries['263']
    try:
        bm25_res = bm25_retriever.search(q_dict['text'])
    except:
        continue
    doc_dict = q_dict['docs']
    state_flag = False
    for doc in bm25_res.iloc:
        if doc.docno in doc_dict:
            state_flag = True
            break
    if state_flag:
        q_dict['bm25'] = bm25_res
        queries2[qid] = q_dict

In [11]:
def get_features(doc, query):
    def title_query_term_overlap(doc, query):
        title_terms = set(doc.title.split())
        query_terms = set(query.split())
        return len(title_terms & query_terms)

    def title_query_jaccard_similarity(doc, query):
        title_terms = set(doc.title.split())
        query_terms = set(query.split())
        intersection = len(title_terms & query_terms)
        union = len(title_terms | query_terms)
        return intersection / union if union else 0

    def title_query_dice_similarity(doc, query):
        title_terms = set(doc.title.split())
        query_terms = set(query.split())
        intersection = len(title_terms & query_terms)
        return (2 * intersection) / (len(title_terms) + len(query_terms)) if title_terms and query_terms else 0

    def title_query_position(doc, query):
        title_words = doc.title.split()
        query_terms = query.split()
        for i, word in enumerate(title_words):
            if word in query_terms:
                return i
        return -1

    def exact_match_title_query(doc, query):
        return 1 if doc.title.strip().lower() == query.strip().lower() else 0

    def query_length(query):
        return len(query.split())

    def query_character_length(query):
        return len(query)

    def term_overlap(doc, query):
        query_terms = set(query.split())
        doc_terms = set(doc.text.split())
        return len(query_terms & doc_terms)

    def jaccard_similarity(doc, query):
        query_terms = set(query.split())
        doc_terms = set(doc.text.split())
        intersection = len(query_terms & doc_terms)
        union = len(query_terms | doc_terms)
        return intersection / union if union else 0

    def dice_similarity(doc, query):
        query_terms = set(query.split())
        doc_terms = set(doc.text.split())
        intersection = len(query_terms & doc_terms)
        return (2 * intersection) / (len(query_terms) + len(doc_terms)) if query_terms and doc_terms else 0

    def term_overlap_extra(doc, query):
        query_terms = set(query.split())
        doc_terms = set(doc.extra_text.split())
        return len(query_terms & doc_terms)

    def jaccard_similarity_extra(doc, query):
        query_terms = set(query.split())
        doc_terms = set(doc.extra_text.split())
        intersection = len(query_terms & doc_terms)
        union = len(query_terms | doc_terms)
        return intersection / union if union else 0

    def dice_similarity_extra(doc, query):
        query_terms = set(query.split())
        doc_terms = set(doc.extra_text.split())
        intersection = len(query_terms & doc_terms)
        return (2 * intersection) / (len(query_terms) + len(doc_terms)) if query_terms and doc_terms else 0









    def document_length(doc):
        return len(doc.text.split())

    def document_character_length(doc):
        return len(doc.text)

    def average_sentence_length(doc):
        sentences = re.split(r'[.!?]', doc.text)
        sentences = [sent.strip() for sent in sentences if sent.strip()]  # Remove empty sentences and leading/trailing spaces
        return sum(len(sent.split()) for sent in sentences) / len(sentences) if sentences else 0

    def stopword_proportion(doc):
        words = doc.split()
        stopword_count = sum(1 for word in words if word.lower() in STOPWORDS)
        return stopword_count / len(words) if words else 0

    def unique_word_count(doc):
        return len(set(doc.text.split()))


    def url_depth(doc):
        return doc.url.count('/')

    def has_query_parameters(doc):
        return '?' in doc.url



    return [
        title_query_term_overlap(doc, query),
        title_query_jaccard_similarity(doc, query),
        title_query_dice_similarity(doc, query),
        title_query_position(doc, query),
        exact_match_title_query(doc, query),
        query_length(query),
        query_character_length(query),
        term_overlap(doc, query),
        jaccard_similarity(doc, query),
        dice_similarity(doc, query),
        term_overlap_extra(doc, query),
        jaccard_similarity_extra(doc, query),
        dice_similarity_extra(doc, query),
        document_length(doc),
        document_character_length(doc),
        average_sentence_length(doc),
        unique_word_count(doc),
        url_depth(doc),
        has_query_parameters(doc),
    ]

In [12]:
for qid, q_dict in queries2.items():
    q_text = q_dict['text']
    q_dict['features'] = []
    for row in q_dict['bm25'].iloc:
        bm25_score = row['score']
        docindex = int(row['docid'])
        doc = dataset.docs[docindex]
        query = q_dict['text']
        qd_features = get_features(doc, query)
        qd_features.append(bm25_score)
        q_dict['features'].append(qd_features)

In [13]:
for qid, q_dict in queries2.items():
    q_dict['scores'] = []
    for row in q_dict['bm25'].iloc:
        score = 0
        if row.docno in q_dict['docs']:
            score = q_dict['docs'][row.docno]
        q_dict['scores'].append(score)

In [14]:
def dcg_at_k(relevances, k=None):
    if k:
        relevances = relevances[:k]
    return sum([rel / np.log2(i + 1) for i, rel in enumerate(relevances, 1)])

def ndcg_at_k(relevances, k=None):
    dcg = dcg_at_k(relevances, k)
    idcg = dcg_at_k(sorted(relevances, reverse=True), k)
    return dcg / idcg if idcg > 0 else 0

In [15]:
def kendalls_tau(f1_index, f2_index, features_list):
    concordont_count = 0
    discordont_count = 0
    n = len(features_list)
    for d1 in range(n):
        for d2 in range(d1+1, n):
            if (
                    (
                        (features_list[d1][f1_index] > features_list[d2][f1_index])
                        and
                        (features_list[d1][f2_index] > features_list[d2][f2_index])
                    ) or
                    (
                        (features_list[d1][f1_index] < features_list[d2][f1_index])
                        and
                        (features_list[d1][f2_index] < features_list[d2][f2_index])
                    )
            ):
                concordont_count += 1
            elif (
                    (
                        (features_list[d1][f1_index] > features_list[d2][f1_index])
                        and
                        (features_list[d1][f2_index] < features_list[d2][f2_index])
                    ) or
                    (
                        (features_list[d1][f1_index] < features_list[d2][f1_index])
                        and
                        (features_list[d1][f2_index] > features_list[d2][f2_index])
                    )
            ):
                discordont_count += 1
    return (concordont_count - discordont_count) / (n*(n-1)/2)

In [16]:
def get_sim_key(f1, f2):
    return str(min(f1, f2)) + "#" + str(max(f1, f2))

In [17]:
for qid,q_dict in queries2.items():
    similarity_dict = q_dict['similarity_dict'] = {}
    feature_count = len(q_dict['features'][0])
    for f1_index in range(feature_count):
        for f2_index in range(f1_index + 1, feature_count):
            key = get_sim_key(f1_index, f2_index)
            similarity_dict[key] = kendalls_tau(f1_index, f2_index, q_dict['features'])

In [23]:
for qid,q_dict in queries2.items():
    n = len(q_dict['features'][0])

    q_dict['feature_scores'] = []
    for f_index in range(feature_count):
        c_feature_list = [
            (
                feature_list[f_index],
                score
            ) for feature_list, score in zip(q_dict['features'], q_dict['scores'])
        ]
        feature_sorted_list = [second for _, second in sorted(c_feature_list, key=lambda x: x[0])]
        ndcg_score = ndcg_at_k(feature_sorted_list)
        feature_sorted_list.reverse()
        ndcg_score2 = ndcg_at_k(feature_sorted_list)
        q_dict['feature_scores'].append(max(ndcg_score,ndcg_score2))


In [59]:
def get_best_feature(features):
    best_feature = None
    best_feature_index = None
    for index, feature in features:
        if not best_feature or feature >= best_feature:
            best_feature = feature
            best_feature_index = index 
    return best_feature_index, best_feature

def update_features(features, best_feature_index, similarity_dict):
    for feature_tuple in features:
        feature_index, feature_score = feature_tuple
        if feature_index != best_feature_index:
            key = get_sim_key(feature_index, best_feature_index)
            sim_val = similarity_dict[key]
            feature_tuple[1] -= sim_val

def GAS(features, similarity_dict):
    features2 = [[index, score] for index, score in enumerate(features)]
    n = len(features2)
    features_ordered = []
    for i in range(n-1):
        best_feature_index, best_feature_score = get_best_feature(features2)
        print(best_feature_index, best_feature_score)
        features_ordered.append(best_feature_index)
        update_features(features2, best_feature_index, similarity_dict)
        features2 = [feature_tuple for feature_tuple in features2 if feature_tuple[0] != best_feature_index]
    features_ordered.append(features2[0][0])
    return features_ordered

In [64]:
for qid, q_dict in queries2.items():
    features_ordered = GAS(q_dict['feature_scores'], q_dict['similarity_dict'])
    q_dict['feature_order'] = features_ordered

9 0.3562071871080222
18 0.2743667067543653
19 0.29275054513820364
15 0.29770557580302354
6 0.26264953503719357
5 0.26264953503719357
4 0.26264953503719357
17 0.23152467845944288
3 0.2577516950047566
12 0.16606060606060602
0 0.05270118995425151
16 -0.05230019549291262
2 -0.08366244640938487
1 -0.20043012317706166
11 -0.674949494949495
7 -0.9490453381445031
10 -1.2808080808080806
14 -1.4046464646464647
8 -1.8961160452152102

[[13, np.float64(-2.56256821763752)]]
19 1.0
6 1.0
5 1.0
4 1.0
2 0.9848484848484849
18 0.9844444444444445
3 0.9363636363636363
17 0.8927130172995494
7 0.926060606060606
1 0.8226262626262626
0 0.7448484848484849
10 0.6571717171717171
12 0.5416161616161619
14 0.5058818582695167
9 0.44909090909090893
11 0.36787878787878814
8 0.20404040404040388
15 0.16791011453548055
16 -0.39282828282828264

[[13, np.float64(-1.335050505050505)]]
12 0.6309297535714574
17 0.3901010101010101
18 0.3716264929473449
2 0.4032529858946897
13 0.3131638089012524
9 0.341010101010101
6 0.301029995